# Implementing LMIR and optimizing the HyperParameter(lambda)

## Step 1:- Loading the data

In [1]:
import json
import csv
import os

# Define file paths
corpus_path = "scifact/corpus.jsonl"
queries_path = "scifact/queries.jsonl"
qrels_train_path = "scifact/qrels/train.tsv"
qrels_test_path = "scifact/qrels/test.tsv"

# 1. Load the Corpus (All documents)
corpus = {}
with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc['_id']] = doc

# 2. Load the Queries (All search terms)
queries = {}
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        query = json.loads(line)
        queries[query['_id']] = query['text']

# 3. Helper function to load a qrels file
def load_qrels(file_path):
    qrels_dict = {}
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found.")
        return qrels_dict
        
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        next(reader) # Skip the header row (query-id, corpus-id, score)
        for row in reader:
            query_id, corpus_id, score = row
            if query_id not in qrels_dict:
                qrels_dict[query_id] = {}
            qrels_dict[query_id][corpus_id] = int(score)
    return qrels_dict

# Load train and test qrels separately
qrels_train = load_qrels(qrels_train_path)
qrels_test = load_qrels(qrels_test_path)

# Combine them into a single master evaluation dictionary
qrels_all = {**qrels_train, **qrels_test}

# Print summary diagnostics
print(f"--- Dataset Statistics ---")
print(f"Total documents loaded in Corpus : {len(corpus)}")
print(f"Total queries loaded            : {len(queries)}")
print(f"Queries with TRAIN answer keys  : {len(qrels_train)}")
print(f"Queries with TEST answer keys   : {len(qrels_test)}")
print(f"Total queries with answer keys  : {len(qrels_all)}")

--- Dataset Statistics ---
Total documents loaded in Corpus : 5183
Total queries loaded            : 1109
Queries with TRAIN answer keys  : 809
Queries with TEST answer keys   : 300
Total queries with answer keys  : 1109


## Step 2:- Text Preprocessing

This step involves:
* **Lowercasing:** 
* **Removing Punctuation**
* **Tokenization:** 

In [2]:
import re

# 1. Define our text cleaning pipeline
def preprocess_text(text):
    if not text:
        return []
        
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    
    return tokens

# 2. Process the Corpus
tokenized_corpus = {}
for doc_id, doc in corpus.items():
    full_text = doc.get('title', '') + " " + doc.get('text', '')
    tokenized_corpus[doc_id] = preprocess_text(full_text)

# 3. Process the Queries
tokenized_queries = {}
for query_id, text in queries.items():
    tokenized_queries[query_id] = preprocess_text(text)

# 4. Verify the results
sample_query_id = list(queries.keys())[0]
print("--- Preprocessing Verification ---")
print("ORIGINAL QUERY:")
print(queries[sample_query_id])
print("\nTOKENIZED QUERY:")
print(tokenized_queries[sample_query_id])

--- Preprocessing Verification ---
ORIGINAL QUERY:
0-dimensional biomaterials lack inductive properties.

TOKENIZED QUERY:
['0dimensional', 'biomaterials', 'lack', 'inductive', 'properties']


## Step 3:- Modeling

#### Storing the Required values for calculating LMIR

In [3]:
from collections import Counter

# Stores tf(term, document)
document_term_frequencies = {}
# Stores |d|
document_lengths = {}
# Stores cf(term)
collection_frequencies = Counter()
# Stores |C|
collection_length = 0

for doc_id, tokens in tokenized_corpus.items():
    # Term frequencies in this document
    term_freqs = Counter(tokens)
    document_term_frequencies[doc_id] = term_freqs
    document_lengths[doc_id] = len(tokens)

    # Update collection statistics
    collection_frequencies.update(tokens)
    collection_length += len(tokens)

print("Collection statistics built successfully.")
print(f"Number of documents      : {len(document_term_frequencies)}")
print(f"Collection length        : {collection_length}")
print(f"Vocabulary size          : {len(collection_frequencies)}")

Collection statistics built successfully.
Number of documents      : 5183
Collection length        : 1106819
Vocabulary size          : 50676


#### Function for Calculating LMIR score

In [4]:
import math

def lmir_jm_score(
        query_tokens,
        doc_id,
        lambda_param=0.2
):
    score = 0.0

    doc_tf = document_term_frequencies[doc_id]
    doc_len = document_lengths[doc_id]

    for term in query_tokens:

        tf = doc_tf.get(term, 0)

        collection_prob = (
            collection_frequencies.get(term, 0)
            / collection_length
        )

        term_prob = (
            (1 - lambda_param) * (tf / doc_len)
            + lambda_param * collection_prob
        )

        if term_prob > 0:
            score += math.log(term_prob)

    return score

#### Document Retrieval Function

In [5]:
def retrieve_lmir_jm(
        tokenized_queries,
        corpus_ids,
        top_k=10,
        lambda_param=0.2
):
    rankings = {}

    for query_id, query_tokens in tokenized_queries.items():

        scores = []

        for doc_id in corpus_ids:

            score = lmir_jm_score(
                query_tokens,
                doc_id,
                lambda_param
            )

            scores.append((doc_id, score))

        scores.sort(
            key=lambda x: x[1],
            reverse=True
        )

        rankings[query_id] = scores[:top_k]

    return rankings

## Step 4:- Evaluation

To determine which search engine performs better, we run both predicted top-10 lists against a human-curated answer key (`qrels_all`). We evaluate performance globally using three core metrics averaged across all queries:

1.  **Precision@10:** Measures precision/noise. Out of the 10 documents returned by the engine, what percentage are actually marked as relevant in the answer key? 
2.  **Recall@10:** Measures comprehensiveness/coverage. Out of all the relevant documents that exist for a query in the database, what percentage did our engine manage to catch within its top 10 slots?
3.  **NDCG@10 (Normalized Discounted Cumulative Gain):** Measures rank order quality. It evaluates whether the engine placed the absolute best, highly relevant documents at the very top (Ranks 1, 2, 3) rather than burying them down at Rank 10, penalizing poor placement logarithmically.

In [6]:
import math

def evaluate_rankings(rankings, qrels, k=10):
    total_queries = 0
    sum_precision = 0.0
    sum_recall = 0.0
    sum_ndcg = 0.0

    for query_id, true_docs in qrels.items():
        if query_id not in rankings:
            continue

        predicted_docs = [doc_id for doc_id, score in rankings[query_id][:k]]
        total_relevant = len(true_docs)

        if total_relevant == 0:
            continue

        total_queries += 1
        relevant_retrieved = 0
        dcg = 0.0

        # 1. Calculate Precision, Recall, and DCG
        for rank_idx, doc_id in enumerate(predicted_docs):
            if doc_id in true_docs and true_docs[doc_id] > 0:
                relevant_retrieved += 1
                relevance_score = true_docs[doc_id]
                
                # DCG Formula: Relevance / log2(Rank + 1)
                # rank_idx is 0-based, so rank_idx + 2 ensures Rank 1 divides by log2(2)
                dcg += relevance_score / math.log2(rank_idx + 2)

        precision_at_k = relevant_retrieved / k
        recall_at_k = relevant_retrieved / total_relevant

        # 2. Calculate Ideal DCG (IDCG)
        ideal_relevance_scores = sorted(true_docs.values(), reverse=True)
        idcg = 0.0
        for rank_idx, rel in enumerate(ideal_relevance_scores[:k]):
            idcg += rel / math.log2(rank_idx + 2)

        # NDCG is the ratio of what we got vs. what was ideally possible
        ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

        sum_precision += precision_at_k
        sum_recall += recall_at_k
        sum_ndcg += ndcg_at_k

    return (sum_precision / total_queries), (sum_recall / total_queries), (sum_ndcg / total_queries)

In [8]:
corpus_ids = list(tokenized_corpus.keys())

#### Optimizing the HyperParameter(lambda)

In [9]:
lambda_values = [0.05, 0.1, 0.15, 0.2, 0.25,0.3, 0.35, 0.4, 0.45, 0.5]

results = []
for lam in lambda_values:
    print(f"Evaluating λ = {lam}")
    
    rankings = retrieve_lmir_jm(
        tokenized_queries,
        corpus_ids,
        top_k=10,
        lambda_param=lam
    )
    
    precision, recall, ndcg = evaluate_rankings(
        rankings,
        qrels_all,
        k=10
    )
    
    results.append(
        (lam, precision, recall, ndcg)
    )

Evaluating λ = 0.05
Evaluating λ = 0.1
Evaluating λ = 0.15
Evaluating λ = 0.2
Evaluating λ = 0.25
Evaluating λ = 0.3
Evaluating λ = 0.35
Evaluating λ = 0.4
Evaluating λ = 0.45
Evaluating λ = 0.5


#### Results

In [13]:
print("\nResults")
print("-" * 60)

best_l,best_n = results[0][0],results[0][3]

for lam, p, r, n in results:
    print(
        f"λ={lam:<4} "
        f"P@10={p:.4f} "
        f"R@10={r:.4f} "
        f"NDCG@10={n:.4f}"
    )
    if n>best_n:
        best_l = lam
        best_n = n

print(f"\nBest Parameter : λ={best_l:<4}")


Results
------------------------------------------------------------
λ=0.05 P@10=0.0782 R@10=0.7003 NDCG@10=0.5906
λ=0.1  P@10=0.0793 R@10=0.7100 NDCG@10=0.5977
λ=0.15 P@10=0.0800 R@10=0.7159 NDCG@10=0.6032
λ=0.2  P@10=0.0806 R@10=0.7206 NDCG@10=0.6086
λ=0.25 P@10=0.0811 R@10=0.7251 NDCG@10=0.6116
λ=0.3  P@10=0.0816 R@10=0.7296 NDCG@10=0.6158
λ=0.35 P@10=0.0822 R@10=0.7350 NDCG@10=0.6195
λ=0.4  P@10=0.0825 R@10=0.7371 NDCG@10=0.6214
λ=0.45 P@10=0.0828 R@10=0.7398 NDCG@10=0.6236
λ=0.5  P@10=0.0830 R@10=0.7416 NDCG@10=0.6272

Best Parameter : λ=0.5 


### Conclusion:
The performance of LMIR-Jelinek-Mercer improved consistently as the smoothing parameter λ increased from 0.05 to 0.5. The best results were achieved at λ = 0.5, yielding Precision@10 = 0.0830, Recall@10 = 0.7416, and NDCG@10 = 0.6272. This indicates that giving greater weight to the collection language model helped improve retrieval effectiveness on the SciFact dataset. Overall, λ = 0.5 was the optimal choice among the tested values and provided the highest ranking quality.